In [7]:
# =============================================================================
# LANGGRAPH CALENDAR AGENT (Pure Standalone - Console Output)
# =============================================================================
import os
import asyncio
from datetime import datetime
from typing import Type, List, Annotated
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool
from dateutil.relativedelta import relativedelta
from langchain_google_community.calendar.update_event import CalendarUpdateEvent
from langchain_google_community.calendar.search_events import CalendarSearchEvents
from langchain_google_community.calendar.create_event import CalendarCreateEvent
from langchain_google_community.calendar.delete_event import CalendarDeleteEvent
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# =============================================================================
# PART 1: Custom Tools (Async + Optimized)
# =============================================================================

class SearchInput(BaseModel):
    min_datetime: str = Field(description="Start of search range (YYYY-MM-DD HH:MM:SS)")
    max_datetime: str = Field(description="End of search range (YYYY-MM-DD HH:MM:SS)")

class CustomCalendarSearchTool(BaseTool):
    name: str = "search_appointments"
    description: str = "Searches the primary calendar for existing events to check availability."
    args_schema: Type[BaseModel] = SearchInput

    def _run(self, min_datetime: str, max_datetime: str) -> str:
        return self._invoke_tool(min_datetime, max_datetime)
    
    async def _ainvoke(self, min_datetime: str, max_datetime: str) -> str:
        return await asyncio.to_thread(self._invoke_tool, min_datetime, max_datetime)
    
    def _invoke_tool(self, min_datetime: str, max_datetime: str) -> str:
        fixed_params = {
            "calendars_info": '[{"id": "primary", "timeZone":  "Asia/Calcutta"}]',
            "max_results": 10
        }
        payload = {**fixed_params, "min_datetime": min_datetime, "max_datetime": max_datetime}
        tool = CalendarSearchEvents()
        try:
            return str(tool.invoke(payload))
        except Exception as e:
            return f"Error searching calendar: {str(e)}"


class AppointmentInput(BaseModel):
    summary: str = Field(description="The appointment type (eg: Interior design etc)")
    start_datetime: str = Field(description="Start time (YYYY-MM-DD HH:MM:SS)")
    end_datetime: str = Field(description="End time = Start time + 60 min (YYYY-MM-DD HH:MM:SS)")
    description: str = Field(description="user's name")
    attendees: List[str] = Field(description="User's email addresses")

class CustomCalendarCreateTool(BaseTool):
    name: str = "create_appointment"
    description: str = "Creates a calendar event with fixed settings. Use this after checking availability."
    args_schema: Type[BaseModel] = AppointmentInput

    def _run(self, summary: str, start_datetime: str, end_datetime: str, description: str, attendees: List[str]) -> str:
        return self._invoke_tool(summary, start_datetime, end_datetime, description, attendees)
    
    async def _ainvoke(self, summary: str, start_datetime: str, end_datetime: str, description: str, attendees: List[str]) -> str:
        return await asyncio.to_thread(self._invoke_tool, summary, start_datetime, end_datetime, description, attendees)
    
    def _invoke_tool(self, summary: str, start_datetime: str, end_datetime: str, description: str, attendees: List[str]) -> str:
        fixed_params = {
            "timezone": "Asia/Kolkata",
            "reminders": [{"method": "popup", "minutes": 60}],
            "conference_data": True,
            "color_id": "5"
        }
        payload = {
            **fixed_params, "summary": summary, "start_datetime": start_datetime,
            "end_datetime": end_datetime, "description": description, "attendees": attendees
        }
        tool = CalendarCreateEvent()
        try:
            result = tool.invoke(payload)
            return f"Success: Event '{summary}' created."
        except Exception as e:
            return f"Error creating event: {str(e)}"


class SearchInputByEmail(BaseModel):
    query: str = Field(description="User's Email id")

class CustomCalendarSearchToolByEmail(BaseTool):
    name: str = "search_appointments_by_email"
    description: str = "Searches for events associated with a specific email address."
    args_schema: Type[BaseModel] = SearchInputByEmail

    def _run(self, query: str) -> str:
        return self._invoke_tool(query)
    
    async def _ainvoke(self, query: str) -> str:
        return await asyncio.to_thread(self._invoke_tool, query)
    
    def _invoke_tool(self, query: str) -> str:
        fixed_params = {
            "calendars_info": '[{"id": "primary", "timeZone": "Asia/Kolkata"}]',
            "min_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "max_datetime": (datetime.now() + relativedelta(months=6)).strftime("%Y-%m-%d %H:%M:%S"),
        }
        payload = {**fixed_params, "query": query}
        tool = CalendarSearchEvents()
        try:
            return str(tool.invoke(payload))
        except Exception as e:
            return f"Error searching calendar: {str(e)}"


class UpdateAppointmentInput(BaseModel):
    event_id: str = Field(description="User's id collected from search_appointments_by_email tool")
    summary: str = Field(description="Updated Meeting Title")
    start_datetime: str = Field(description="Start time (YYYY-MM-DD HH:MM:SS)")
    end_datetime: str = Field(description="End time (YYYY-MM-DD HH:MM:SS)")

class CustomCalendarUpdateTool(BaseTool):
    name: str = "update_appointment"
    description: str = "Updates an existing calendar event using the event ID."
    args_schema: Type[BaseModel] = UpdateAppointmentInput

    def _run(self, event_id: str, summary: str, start_datetime: str, end_datetime: str) -> str:
        return self._invoke_tool(event_id, summary, start_datetime, end_datetime)
    
    async def _ainvoke(self, event_id: str, summary: str, start_datetime: str, end_datetime: str) -> str:
        return await asyncio.to_thread(self._invoke_tool, event_id, summary, start_datetime, end_datetime)
    
    def _invoke_tool(self, event_id: str, summary: str, start_datetime: str, end_datetime: str) -> str:
        fixed_params = {"timezone": "Asia/Kolkata", "send_updates": "all"}
        payload = {
            **fixed_params, "event_id": event_id, "summary": summary,
            "start_datetime": start_datetime, "end_datetime": end_datetime,
        }
        tool = CalendarUpdateEvent()
        try:
            result = tool.invoke(payload)
            return f"Success: Event '{summary}' updated."
        except Exception as e:
            return f"Error updating event: {str(e)}"


class DeleteEventByEmail(BaseModel):
    event_id: str = Field(description="User's event id collected from search_appointments_by_email tool")

class DeleteEventTool(BaseTool):
    name: str = "cancel_appointment"
    description: str = "Delete events associated with a specific email address."
    args_schema: Type[BaseModel] = DeleteEventByEmail

    def _run(self, event_id: str) -> str:
        return self._invoke_tool(event_id)
    
    async def _ainvoke(self, event_id: str) -> str:
        return await asyncio.to_thread(self._invoke_tool, event_id)
    
    def _invoke_tool(self, event_id: str) -> str:
        fixed_params = {"send_updates": "all"}
        payload = {**fixed_params, "event_id": event_id}
        tool = CalendarDeleteEvent()
        try:
            return str(tool.invoke(payload))
        except Exception as e:
            return f"Error deleting calendar: {str(e)}"


# =============================================================================
# PART 2: Initialize Tools & LLM
# =============================================================================

tools = [
    CustomCalendarSearchTool(),
    CustomCalendarCreateTool(),
    CustomCalendarSearchToolByEmail(),
    CustomCalendarUpdateTool(),
    DeleteEventTool()
]

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools(tools)


# =============================================================================
# PART 3: State Definition
# =============================================================================

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


# =============================================================================
# PART 4: System Prompt
# =============================================================================

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
system_prompt = f"""
You are an AI Receptionist. You help people by solving their query.
Today's date is: {timestamp}.

================
TOOLS 
================
- Use available tools as needed, or upon user request.
- Collect required inputs first. Perform actions silently if the runtime expects it.
- Speak outcomes clearly. If an action fails, say so once, propose a fallback, or ask how to proceed.
- When tools return structured data, summarize it to the user in a way that is easy to understand, and don't directly recite identifiers or other technical details.
==========================================
 FOR BOOKING AN APPOINTMENT
==========================================

Appointments are 60 minutes long.

**INFORMATION REQUIRED:**
1. Name
2. Appointment Date & Time
3. Appointment Type
4. Email
Ask for missing details politely.

**STEP 1: CHECK AVAILABILITY**
Use the 'search_appointments' tool.
Calculate the 'min_datetime' as the start of the requested day (e.g., 2026-01-25 00:00:00).
Calculate the 'max_datetime' as the end of the requested day (e.g., 2026-01-25 23:59:59).

**STEP 2: ANALYZE**
Check the search results.
If the user's requested time overlaps with an existing event, tell the user it is busy and offer the free times.
If the time is free, FIRST VERIFY with the USER then proceed to booking.

**STEP 3: BOOK**
Use the 'create_appointment' tool with the confirmed details.

==================================================
 FOR UPDATE AN APPOINTMENT
==================================================

Step 1: IDENTIFY APPOINTMENT
- Ask the user for their **Email Address**.
- Use 'search_appointments_by_email' to find their events.

Step 2: CONFIRM DETAILS
- If multiple events are found, list the Summary and Time of each event.
- Ask the user which specific event they want to update.
- Note down the 'id' and the 'summary' (current title) of the chosen event.

Step 3: COLLECT UPDATES
- Ask the user for the **New Date and Time**.
- Ask: 'Do you want to change the appointment type?'

Step 4: CHECK AVAILABILITY
Use the 'search_appointments' tool.

Step 5: ANALYZE
Check the search results.
If the user's requested time overlaps with an existing event, tell the user it is busy and offer the free times.
If the time is free, proceed to booking.

Step 6: PREPARE SUMMARY (IMPORTANT LOGIC)
- If the user provides a NEW title: Use that new title.
- If the user says NO (or does not want to change it): 
  Take the EXISTING title from the search results and append ' updated' to it.
  Example: Existing title 'Meeting' becomes 'Meeting updated'.

Step 7: EXECUTE UPDATE
Use 'update_appointment' with the event ID, the calculated summary, and new times.

===============================
 CANCEL AN APPOINTMENT
===============================

Step 1: IDENTIFY APPOINTMENT
- Ask the user for their **Email Address**.
- Use 'search_appointments_by_email' to find their upcoming events.

Step 2: SELECT EVENT
- If multiple events are found, list the 'Summary' (Title) and 'Time' of each event clearly.
- Ask the user to specify which one they want to cancel (by Title or Time).
- Note the 'id' of the selected event.

Step 3: CONFIRM CANCELLATION
- Before cancelling, confirm with the user by stating the Title and Time.
- Example: 'Are you sure you want to cancel the Meeting on Jan 25 at 11:00?'

Step 4: EXECUTE
- If the user confirms, use 'cancel_appointment' with the event ID.
- If the user says no, ask if they want to cancel a different event.

==========================
CONVERSATIONAL FLOW
==========================

- Help the user accomplish their objective efficiently and correctly. Prefer the simplest safe step first. Check understanding and adapt.
- Provide guidance in small steps and confirm completion before continuing.
- Summarize key results when closing a topic.
"""


# =============================================================================
# PART 5: Nodes with Console Acknowledgments
# =============================================================================

ACTION_ACK_PHRASES = {
    "search_appointments": "🔊 Checking your calendar for availability...",
    "create_appointment": "🔊 Creating your appointment now...",
    "search_appointments_by_email": "🔊 Searching for your events by email...",
    "update_appointment": "🔊 Updating your appointment...",
    "cancel_appointment": "🔊 Cancelling your appointment..."
}


def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    messages_with_system = [SystemMessage(content=system_prompt)] + messages
    response = llm_with_tools.invoke(messages_with_system)
    return {"messages": [response]}


async def chat_node_async(state: ChatState):
    """Async LLM node."""
    messages = state["messages"]
    messages_with_system = [SystemMessage(content=system_prompt)] + messages
    response = await asyncio.to_thread(llm_with_tools.invoke, messages_with_system)
    return {"messages": [response]}


def tool_node_with_console_ack(state: ChatState):
    """
    ToolNode that prints acknowledgment to console BEFORE executing tools.
    Pure standalone approach - no callbacks, no LiveKit dependencies.
    """
    messages = state["messages"]
    last_message = messages[-1] if messages else None
    
    # Extract tool calls from the last AI message
    tool_calls = getattr(last_message, "tool_calls", []) if last_message else []
    
    if tool_calls:
        # Print acknowledgment to console BEFORE tool execution
        for tool_call in tool_calls:
            tool_name = tool_call.get("name", "")
            ack_phrase = ACTION_ACK_PHRASES.get(tool_name, "🔊 One moment please...")
            print(f"\n{ack_phrase}")  # ✅ Direct console output
            break
    
    # Execute tools
    tool_node = ToolNode(tools)
    result = tool_node.invoke(state)
    
    return result


# =============================================================================
# PART 6: Build Graph
# =============================================================================

checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node_async)
graph.add_node("tools", tool_node_with_console_ack)  # ✅ Console acks

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

compiled_graph = graph.compile(checkpointer=checkpointer)


In [8]:
state = {"messages": []}

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chat ended 👋")
        break

    state["messages"].append(
        {"role": "user", "content": user_input}
    )

    print("Assistant: ", end="")

    async for msg, metadata in compiled_graph.astream(
        state,
        config = {"configurable": {"thread_id": "1"}},
        stream_mode="messages"
    ):
        if msg.content:
            print(msg.content, end="", flush=True)

    state["messages"].append(
        {"role": "assistant", "content": msg.content}
    )

    print()

Assistant: 
🔊 Checking your calendar for availability...
[{'id': 'edd7i6v5t1aeau5c303jakvgtc', 'htmlLink': 'https://www.google.com/calendar/event?eid=ZWRkN2k2djV0MWFlYXU1YzMwM2pha3ZndGMgcHJhYmh1ZGF0dGFwYXRyYTI4QG0', 'summary': 'Interior Design', 'creator': 'prabhudattapatra28@gmail.com', 'organizer': 'prabhudattapatra28@gmail.com', 'start': '2026-04-29T22:00:00+05:30', 'end': '2026-04-29T23:00:00+05:30'}]I’ve checked the calendar for today, April 29, 2026. There’s an existing interior‑design appointment from 10:00 PM – 11:00 PM, but the slot **starting at 11:00 PM (23:00) and ending at midnight** is free.

Would you like me to book your interior‑design appointment for **23:00 – 00:00** today?
Assistant: 
🔊 Creating your appointment now...
Success: Event 'Interior Design' created.Your interior‑design appointment has been booked:

- **Name:** Dev  
- **Date & Time:** Today, April 29, 2026 from 11:00 PM to 12:00 AM (midnight)  
- **Type:** Interior Design  
- **Email:** dev23@gmail.com  
